## 编写代码Agent

In [1]:
import os

from bs4 import BeautifulSoup
from langchain_community.document_loaders.recursive_url_loader import RecursiveUrlLoader

In [70]:
# LCEL Docs
url = "https://openpyxl.readthedocs.io/en/stable/"

global x_value

# def extractor_func(x):
#     global x_value
#     if x_value is None:
#         print(type(x))
#         print(x)
#         x_value = x

#     return BeautifulSoup(x, "html.parser").text

# 实例化loader
loader = RecursiveUrlLoader(
    url=url, max_depth=2, extractor=lambda x: BeautifulSoup(x, "html.parser").text
)
# loader = RecursiveUrlLoader("https://docs.python.org/3.9/", max_depth=2, extractor=lambda x: BeautifulSoup(x, "html.parser").text)
# 加载文档
docs = loader.load()

# 排序
docs_sorted = sorted(docs, key=lambda x: x.metadata["source"])
docs_reversed = list(reversed(docs_sorted))
docs_content = "\n\n\n---\n\n\n".join([doc.page_content for doc in docs_reversed])

In [71]:
docs_content

'\n\n\n\n\nTutorial — openpyxl 3.1.3 documentation\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nNavigation\n\n\nindex\n\nmodules |\n\nnext |\n\nprevious |\nopenpyxl 3.1.3 documentation »\nTutorial\n\n\n\n\n\n\n\nTutorial\uf0c1\n\nInstallation\uf0c1\nInstall openpyxl using pip. It is advisable to do this in a Python virtualenv\nwithout system packages:\n$ pip install openpyxl\n\n\n\nNote\nThere is support for the popular lxml library which will be used if it\nis installed. This is particular useful when creating large files.\n\n\nWarning\nTo be able to include images (jpeg, png, bmp,…) into an openpyxl file,\nyou will also need the “pillow” library that can be installed with:\n$ pip install pillow\n\n\nor browse https://pypi.python.org/pypi/Pillow/, pick the latest version\nand head to the bottom of the page for Windows binaries.\n\n\nWorking with a checkout\uf0c1\nSometimes you might want to work with the checkout of a particular version.\nThis may be the case if bugs have been fixed but a rel

In [69]:
len(docs_reversed)

1

In [10]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain.prompts import MessagesPlaceholder
from langchain.prompts import ChatPromptTemplate
from pydantic.v1 import BaseModel, Field
from langchain.chat_models import init_chat_model

In [39]:
system_prompt = """You are a code programming assistant, helping users perform operations related to Excel files. Ensure any code you provide can be executed \n
    with all required imports and variables defined. Structure your answer with a description of the code solution. \n
    Then list the imports. And finally list the functioning code block.
  Here is the user question:
"""

code_gen_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("messages")
    ]
)

In [40]:
code_gen_prompt

ChatPromptTemplate(input_variables=['messages'], input_types={'messages': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchain_core.mes

In [41]:
# DataModel
class Code(BaseModel):
    """
    Code output
    """
    prefix: str = Field(description="Description of the problem and approach")
    imports: str = Field(description="Code block import statements")
    code: str = Field(description="Code block not including import statements")
    description = "Schema for code solutions to questions about LCEL."

# 实例化模型
model_name = "qwen2.5:7b"
# model_name = "qwq:32b"
ollama_api_base = os.environ["OLLAMA_API_BASE"]
model = init_chat_model(model=model_name, model_provider="ollama", base_url=ollama_api_base)

code_gen_chain = code_gen_prompt | model.with_structured_output(Code)

In [45]:
# 现在我们咨询个问题
question = "创建一个excel，名字是xx.xllsx,请在第一个sheet中插入3列：姓名，年龄，手机号。随机插入100条数据"
solution = code_gen_chain.invoke({
    "context": docs_content,
    "messages": [HumanMessage(content=question)]
})
solution

Code(prefix='创建一个名为xx.xlsx的Excel文件，第一个工作表包含姓名、年龄、手机号三列，随机生成100条数据。', imports='import pandas as pd\nimport random', code="names = [f'Name_{i}' for i in range(1, 101)]\nages = [random.randint(18, 60) for _ in range(100)]\nphones = []\nfor _ in range(100):\n    prefix = random.choice(['13', '15', '17', '18', '19'])\n    suffix = ''.join([str(random.randint(0, 9)) for _ in range(9)])\n    phones.append(f'{prefix}{suffix}')\ndf = pd.DataFrame({'姓名': names, '年龄': ages, '手机号': phones})\ndf.to_excel('xx.xlsx', sheet_name='Sheet1', index=False)", description='Schema for code solutions to questions about LCEL.')

In [46]:
solution.prefix

'创建一个名为xx.xlsx的Excel文件，第一个工作表包含姓名、年龄、手机号三列，随机生成100条数据。'

In [47]:
print(solution.imports)

import pandas as pd
import random


In [48]:
print(solution.code)

names = [f'Name_{i}' for i in range(1, 101)]
ages = [random.randint(18, 60) for _ in range(100)]
phones = []
for _ in range(100):
    prefix = random.choice(['13', '15', '17', '18', '19'])
    suffix = ''.join([str(random.randint(0, 9)) for _ in range(9)])
    phones.append(f'{prefix}{suffix}')
df = pd.DataFrame({'姓名': names, '年龄': ages, '手机号': phones})
df.to_excel('xx.xlsx', sheet_name='Sheet1', index=False)
